In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
GRAPH_DIR = PROCESSED_DIR / "graph"

GRAPH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GRAPH_NODES_FILE = PROCESSED_DIR / "graph_nodes.csv"
GRAPH_EDGES_FILE = PROCESSED_DIR / "graph_edges.csv"

NODE_MAPPING_FILE = GRAPH_DIR / "node_mapping.csv"
EDGES_INDEX_FILE = GRAPH_DIR / "edges_index.csv"
GRAPH_STATS_FILE = GRAPH_DIR / "graph_statistics.json"

print("Project root:", PROJECT_ROOT)
print("Graph directory:", GRAPH_DIR)

Project root: c:\Users\Shubham\OneDrive\Desktop\MajorProject
Graph directory: c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\processed\graph


In [2]:
nodes_df = pd.read_csv(
    GRAPH_NODES_FILE,
    usecols=["paperId"]
)

nodes_df["paperId"] = (
    nodes_df["paperId"]
    .astype(str)
    .str.strip()
)

print("Graph nodes:", len(nodes_df))
print("Unique IDs:", nodes_df["paperId"].nunique())

Graph nodes: 92081
Unique IDs: 92081


In [3]:
nodes_df = nodes_df.reset_index(drop=True)

nodes_df["node_id"] = np.arange(
    len(nodes_df),
    dtype=np.int64
)

# Put node_id first
nodes_df = nodes_df[
    ["node_id", "paperId"]
]

print(nodes_df.head())

   node_id                                   paperId
0        0  00000c33779acab142af6c7a6dae8b36fac0805d
1        1  0000238f07f151172cf2602588ba762b55c8464b
2        2  0000315635be19f6278dbc72597b3065fac405f0
3        3  00005d68c6c7eb4d3c27da8242a30b9a498f991e
4        4  0000627a402141da113dfb8148711fecfcb4fdd1


In [4]:
assert nodes_df["node_id"].is_unique
assert nodes_df["paperId"].is_unique

assert nodes_df["node_id"].min() == 0

assert (
    nodes_df["node_id"].max()
    ==
    len(nodes_df) - 1
)

print("Node mapping validation PASSED")

Node mapping validation PASSED


In [5]:
nodes_df.to_csv(
    NODE_MAPPING_FILE,
    index=False,
    encoding="utf-8"
)

print(
    "Saved:",
    NODE_MAPPING_FILE
)

Saved: c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\processed\graph\node_mapping.csv


In [6]:
edges_df = pd.read_csv(
    GRAPH_EDGES_FILE,
    usecols=[
        "source_paper",
        "target_paper",
        "relationship"
    ]
)

edges_df["source_paper"] = (
    edges_df["source_paper"]
    .astype(str)
    .str.strip()
)

edges_df["target_paper"] = (
    edges_df["target_paper"]
    .astype(str)
    .str.strip()
)

print("Original graph edges:", len(edges_df))

Original graph edges: 103482


In [7]:
paper_to_node = dict(
    zip(
        nodes_df["paperId"],
        nodes_df["node_id"]
    )
)

print(
    "Mapping size:",
    len(paper_to_node)
)

Mapping size: 92081


In [8]:
edges_df["source_id"] = (
    edges_df["source_paper"]
    .map(paper_to_node)
)

edges_df["target_id"] = (
    edges_df["target_paper"]
    .map(paper_to_node)
)

In [9]:
missing_source = edges_df["source_id"].isna().sum()
missing_target = edges_df["target_id"].isna().sum()

print("Missing source mappings:", missing_source)
print("Missing target mappings:", missing_target)

assert missing_source == 0
assert missing_target == 0

print("Edge mapping validation PASSED")

Missing source mappings: 0
Missing target mappings: 0
Edge mapping validation PASSED


In [10]:
forward_edges = edges_df[
    ["source_id", "target_id"]
].copy()

reverse_edges = edges_df[
    ["target_id", "source_id"]
].copy()

reverse_edges.columns = [
    "source_id",
    "target_id"
]

edge_index_df = pd.concat(
    [
        forward_edges,
        reverse_edges
    ],
    ignore_index=True
)

print(
    "Directed citation edges:",
    len(forward_edges)
)

print(
    "Bidirectional message-passing edges:",
    len(edge_index_df)
)

Directed citation edges: 103482
Bidirectional message-passing edges: 206964


In [11]:
before = len(edge_index_df)

edge_index_df = edge_index_df.drop_duplicates(
    subset=[
        "source_id",
        "target_id"
    ]
).reset_index(drop=True)

after = len(edge_index_df)

print("Before deduplication:", before)
print("After deduplication:", after)
print(
    "Duplicates removed:",
    before - after
)

Before deduplication: 206964
After deduplication: 206964
Duplicates removed: 0


In [12]:
self_loops = (
    edge_index_df["source_id"]
    ==
    edge_index_df["target_id"]
).sum()

print(
    "Self-loops:",
    self_loops
)

assert self_loops == 0

Self-loops: 0


In [13]:
num_nodes = len(nodes_df)

invalid_source = (
    (edge_index_df["source_id"] < 0)
    |
    (edge_index_df["source_id"] >= num_nodes)
).sum()

invalid_target = (
    (edge_index_df["target_id"] < 0)
    |
    (edge_index_df["target_id"] >= num_nodes)
).sum()

print("Invalid source IDs:", invalid_source)
print("Invalid target IDs:", invalid_target)

assert invalid_source == 0
assert invalid_target == 0

Invalid source IDs: 0
Invalid target IDs: 0


In [14]:
edge_index_df.to_csv(
    EDGES_INDEX_FILE,
    index=False,
    encoding="utf-8"
)

print(
    "Saved:",
    EDGES_INDEX_FILE
)

print(
    "Total message-passing edges:",
    len(edge_index_df)
)

Saved: c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\processed\graph\edges_index.csv
Total message-passing edges: 206964


In [15]:
degree_counts = np.zeros(
    num_nodes,
    dtype=np.int64
)

degree_counts += np.bincount(
    edge_index_df["source_id"].to_numpy(),
    minlength=num_nodes
)

print("=" * 70)
print("GRAPH PREPARATION STATISTICS")
print("=" * 70)

print(
    "Nodes:",
    f"{num_nodes:,}"
)

print(
    "Original citation edges:",
    f"{len(edges_df):,}"
)

print(
    "Message-passing edges:",
    f"{len(edge_index_df):,}"
)

print(
    "Average degree:",
    round(degree_counts.mean(), 2)
)

print(
    "Median degree:",
    int(np.median(degree_counts))
)

print(
    "Maximum degree:",
    int(degree_counts.max())
)

print(
    "Degree 1:",
    int((degree_counts == 1).sum())
)

print(
    "Degree >= 5:",
    int((degree_counts >= 5).sum())
)

print(
    "Degree >= 10:",
    int((degree_counts >= 10).sum())
)

GRAPH PREPARATION STATISTICS
Nodes: 92,081
Original citation edges: 103,482
Message-passing edges: 206,964
Average degree: 2.25
Median degree: 1
Maximum degree: 383
Degree 1: 83134
Degree >= 5: 3136
Degree >= 10: 2497


In [16]:
graph_stats = {
    "num_nodes": int(num_nodes),
    "original_citation_edges": int(len(edges_df)),
    "message_passing_edges": int(len(edge_index_df)),
    "average_degree": float(degree_counts.mean()),
    "median_degree": float(np.median(degree_counts)),
    "maximum_degree": int(degree_counts.max()),
    "degree_1_nodes": int(
        (degree_counts == 1).sum()
    ),
    "degree_ge_5_nodes": int(
        (degree_counts >= 5).sum()
    ),
    "degree_ge_10_nodes": int(
        (degree_counts >= 10).sum()
    )
}

with open(
    GRAPH_STATS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        graph_stats,
        f,
        indent=4
    )

print(
    "Saved:",
    GRAPH_STATS_FILE
)

Saved: c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\processed\graph\graph_statistics.json


In [17]:
print()
print("=" * 70)
print("GRAPH PREPARATION COMPLETE")
print("=" * 70)

print()

print(
    f"Node mapping:       {NODE_MAPPING_FILE}"
)

print(
    f"Edge index:          {EDGES_INDEX_FILE}"
)

print(
    f"Statistics:          {GRAPH_STATS_FILE}"
)

print()

print(
    f"Nodes:               {num_nodes:,}"
)

print(
    f"Original edges:      {len(edges_df):,}"
)

print(
    f"Message edges:       {len(edge_index_df):,}"
)

print()

print("All graph validation checks passed.")


GRAPH PREPARATION COMPLETE

Node mapping:       c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\processed\graph\node_mapping.csv
Edge index:          c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\processed\graph\edges_index.csv
Statistics:          c:\Users\Shubham\OneDrive\Desktop\MajorProject\data\processed\graph\graph_statistics.json

Nodes:               92,081
Original edges:      103,482
Message edges:       206,964

All graph validation checks passed.
